# 002 · Variables and Names

> **Syntax › Basics** · Python 3.14

In C# a variable is a typed storage location you declare up front. In Python a variable is just a **name bound to an object** — no declaration, no type, created the first time you assign to it.

**In this notebook you will learn:**

- how assignment binds names to objects (and why there is no `var`, `int` or declaration)
- dynamic vs. static typing, and rebinding a name to a different type
- multiple assignment and the one-line swap `a, b = b, a`
- naming conventions (PEP 8) compared with C#'s, and how constants work without `const`
- `del`, `NameError`, reserved keywords, and the `_` convention

## C# mental model

| C# | Python |
|---|---|
| `int count = 0;` / `var count = 0;` | `count = 0` |
| static typing — a variable's type is fixed at compile time | dynamic typing — a *name* has no type; the *object* it refers to does |
| using an unassigned variable is a compile error | using an unbound name is a runtime `NameError` |
| `const int Max = 10;` | `MAX = 10` (convention only) or `MAX: Final = 10` (checked by type checkers) |
| `PascalCase` methods, `camelCase` locals | `snake_case` functions and variables |
| `(a, b) = (b, a);` | `a, b = b, a` |

The key mental shift: think of every Python variable as a C# `object` reference. Assignment never copies a value — it makes a name *point at* an object.

## Assignment creates the name

```csharp
// C#
int count = 10;
var message = "hello";
string? later;          // declared, not yet assigned
```

Python has no declaration statement. The first assignment creates the name; there is nothing like `string? later;` to reserve a name without a value.

In [ ]:
count = 10
message = "hello"
price = 9.99

print(count, message, price)
print(type(count), type(message), type(price))   # the *objects* have types

You *can* add a type annotation, which looks a bit like a C# declaration. It documents intent for tools such as mypy/pyright, but Python does **not** enforce it at runtime *(see Type Hints › 001 · Type Annotations)*.

In [ ]:
total: int = 0
total = "not an int"    # runs fine — only a type checker would complain
print(total)

## Names refer to objects

Assignment binds a name to an object. Assigning one name to another makes **both names refer to the same object** — exactly like copying a reference-type variable in C#. The built-in `id()` returns an object's identity (in CPython, its memory address).

In [ ]:
a = [1, 2, 3]
b = a                 # no copy: b now refers to the same list object
b.append(4)

print(a)              # [1, 2, 3, 4] — the change is visible through both names
print(a is b)         # True — same object (C#: ReferenceEquals(a, b))
print(id(a) == id(b))

For immutable objects such as `int` and `str` this sharing is harmless, because they can never change — "modifying" them really means binding the name to a **new** object:

In [ ]:
x = 10
y = x
x = x + 1     # creates a new int object and rebinds x; y still points at 10

print(x, y)   # 11 10

Mutability and identity get a full treatment later *(see Data Types › 007 · Mutability and Identity)*.

## Dynamic typing: rebinding to a different type

In C#, `var` is still statically typed — the compiler infers the type once and it can never change:

```csharp
// C#
var value = 42;
value = "forty-two";   // compile error CS0029: cannot convert string to int
```

In Python the name is not tied to any type, so you can rebind it to anything:

In [ ]:
value = 42
print(value, type(value).__name__)

value = "forty-two"
print(value, type(value).__name__)

value = [4, 2]
print(value, type(value).__name__)

⚠️ **Gotcha:** "dynamic" does not mean "weak". Python is **strongly** typed — it will not silently convert between unrelated types the way JavaScript does. Mixing a `str` and an `int` is a `TypeError`:

In [ ]:
try:
    result = "Total: " + 5
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print("Total: " + str(5))   # convert explicitly
print(f"Total: {5}")        # or use an f-string

Reusing one name for values of different types is legal but confusing. In real code, keep each name to one meaning (and one type).

## Multiple assignment and swapping

You can assign several names at once. The right-hand side is evaluated **completely first**, then the values are bound from left to right — which is what makes the one-line swap work without a temporary variable.

```csharp
// C# 7+ — tuple deconstruction
var (x, y) = (1, 2);
(x, y) = (y, x);       // swap
```

In [ ]:
x, y = 1, 2
print(x, y)

x, y = y, x            # swap — no temp variable needed
print(x, y)

In [ ]:
# Chained assignment: all names refer to the SAME object
a = b = c = 0
print(a, b, c)

# ⚠️ careful with mutable objects — this is ONE list with two names:
first = second = []
first.append("oops")
print(second)          # ['oops']

This "unpacking" works with any iterable, and has many more tricks (`*rest`, nested patterns) *(see Collections › 006 · Unpacking)*.

## Naming conventions (PEP 8)

PEP 8 is Python's official style guide. Its naming rules differ from .NET's, and Python developers follow them closely — tools like `ruff` will flag violations.

| Kind | C# convention | Python convention (PEP 8) |
|---|---|---|
| Local variable / parameter | `orderTotal` | `order_total` |
| Method / function | `CalculateTotal()` | `calculate_total()` |
| Property / field | `OrderTotal` / `_orderTotal` | `order_total` / `_order_total` |
| Class | `OrderService` | `OrderService` (same!) |
| Constant | `MaxRetries` (`const`) | `MAX_RETRIES` |
| Interface | `IRepository` | no `I` prefix — `Repository` (often a `Protocol`) |
| Module / file | `OrderService.cs` | `order_service.py` |

In [ ]:
max_retries = 3                 # snake_case variable
user_name = "alice"

def calculate_total(prices):    # snake_case function
    return sum(prices)

class ShoppingCart:             # PascalCase class (same as C#)
    pass

print(calculate_total([1.5, 2.5]), user_name, ShoppingCart.__name__)

Names are case-sensitive (like C#), may contain letters, digits and underscores, and must not start with a digit. Unicode letters are allowed, but stick to ASCII in practice.

## Constants: no `const`

Python has no `const` or `readonly` keyword. A name in `UPPER_CASE` at module level is a **convention** that tells readers "do not rebind this" — nothing stops you.

```csharp
// C#
public const int MaxRetries = 3;
MaxRetries = 4;   // compile error
```

In [ ]:
MAX_RETRIES = 3
MAX_RETRIES = 4          # allowed at runtime — the convention is the only guard
print(MAX_RETRIES)

For tool-enforced constants, annotate with `typing.Final`. The interpreter still allows reassignment, but mypy/pyright report it as an error — similar in spirit to a C# compiler error, just provided by a separate tool.

In [ ]:
from typing import Final

TIMEOUT_SECONDS: Final = 30
# TIMEOUT_SECONDS = 60   # a type checker reports: cannot assign to final name
print(TIMEOUT_SECONDS)

## Deleting a name with `del`

`del name` removes the **binding** (the name), not necessarily the object — the object is garbage-collected once nothing else refers to it. There is no C# equivalent; the closest idea is a variable going out of scope.

Using a name that is not bound raises `NameError` at runtime (in C# this would be a compile-time error).

In [ ]:
temp = "scratch value"
alias = temp
del temp                 # removes the name 'temp' only

print(alias)             # the object is still alive through 'alias'

try:
    print(temp)
except NameError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** a typo in a variable name is not caught until that line actually runs — and an assignment with a typo silently creates a *new* variable:

In [ ]:
total_price = 100
totl_price = total_price * 2   # typo creates a brand-new name, no error
print(total_price)             # still 100 — the bug is silent

Linters (ruff, pylint) and type checkers catch most of these; use them the way you rely on the C# compiler.

## Keywords can't be names

Python has a small set of reserved keywords. Unlike C#, there is no `@` escape (`@class`) — the convention is to append an underscore instead: `class_`, `from_`, `type_`.

In [ ]:
import keyword

print(len(keyword.kwlist), "hard keywords:")
print(keyword.kwlist)

try:
    compile("class = 'economy'", "<example>", "exec")
except SyntaxError as e:
    print(f"{type(e).__name__}: {e}")

class_ = "economy"       # PEP 8 convention: trailing underscore
print(class_)

*Soft keywords* (`match`, `case`, `type`, `_`) are keywords only in specific positions, so they remain valid names elsewhere — a bit like C#'s contextual keywords (`var`, `value`, `record`).

In [ ]:
print(keyword.softkwlist)

match = "a valid variable name"   # legal, though best avoided
type_name = type(match).__name__
print(match, "->", type_name)

⚠️ **Gotcha:** built-in function names like `list`, `str`, `id`, `type`, `sum` and `input` are **not** keywords, so you can accidentally shadow them:

In [ ]:
list = [1, 2, 3]          # shadows the built-in list type!
try:
    list("abc")
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

del list                  # remove our name; the built-in becomes visible again
print(list("abc"))

## The `_` convention

A single underscore is an ordinary name that, by convention, means **"I don't care about this value"** — like C#'s discard `_`. In C# `_` is a true discard; in Python it is a real variable that simply gets overwritten.

```csharp
// C#
var (name, _) = GetPair();
for (var i = 0; i < 3; i++) { /* index unused */ }
```

In [ ]:
for _ in range(3):             # loop variable not needed
    print("hi")

name, _ = ("alice", 42)        # ignore the second value
print(name)

Other underscore conventions you will meet later:

- `_name` — "internal/private by convention" (like `private`, but not enforced) *(see Classes & OOP › 004 · Encapsulation and Naming Conventions)*
- `__name` — triggers name mangling inside classes
- `__name__` — "dunder" names reserved for Python's special methods and attributes *(see Classes & OOP › 007 · Dunder Methods)*
- `1_000_000` — underscores as digit separators in numeric literals (like C#'s `1_000_000`)

In the interactive REPL, `_` also holds the result of the last expression.

## Summary

| Task | C# | Python |
|---|---|---|
| Declare and assign | `int n = 1;` / `var n = 1;` | `n = 1` |
| Annotate type | `int n = 1;` (enforced) | `n: int = 1` (not enforced) |
| Change type | not allowed | rebind freely: `n = "one"` |
| Multiple assignment | `var (a, b) = (1, 2);` | `a, b = 1, 2` |
| Swap | `(a, b) = (b, a);` | `a, b = b, a` |
| Constant | `const int Max = 10;` | `MAX = 10` / `MAX: Final = 10` |
| Same object? | `ReferenceEquals(a, b)` | `a is b` |
| Remove a variable | (goes out of scope) | `del name` |
| Keyword as name | `@class` | `class_` |
| Discard | `_` | `_` (a real name, by convention) |

## Exercises

1. Given `left, middle, right = "a", "b", "c"`, rotate the values in **one statement** so that you get `"b", "c", "a"`.
2. Rewrite these C#-style names in PEP 8 style: `MaxItemCount` (a constant), `GetUserName` (a function), `orderTotal` (a variable), `IOrderRepository` (an interface-like class).
3. Predict the output, then run it:
   ```python
   a = [1]
   b = a
   a = a + [2]
   print(a, b)
   ```

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1 — the right-hand side is evaluated first, then unpacked
left, middle, right = "a", "b", "c"
left, middle, right = middle, right, left
print(left, middle, right)   # b c a

# 2
MAX_ITEM_COUNT = 100
def get_user_name(): ...
order_total = 0
class OrderRepository: ...   # no "I" prefix in Python

# 3 — prints [1, 2] [1]
# `a + [2]` builds a NEW list and rebinds `a` to it; `b` still refers to the
# original list. (Compare with a.append(2) or a += [2], which mutate in place.)
```

</details>